# FIAP TECH CHALLENGE 1 - GRUPO SSP

## 2. ANÁLISE DO DICIONÁRIO DE DADOS - SINAN VIOL

Este notebook apresenta uma análise sucinta de cada coluna do dataset SINAN - Violência Interpessoal/Autoprovocada, baseado no dicionário de dados oficial do Ministério da Saúde.

**Fonte:** [Dicionário de Dados SINAN NET - Versão 5.0/Patch 5.1](docs/dicionario_dados_sinan.pdf)

**Dataset:** VIOLBR24 (Violência - Brasil - 2024)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Função para encontrar a raiz do projeto
def find_project_root():
    """Sobe diretórios até encontrar a raiz do projeto."""
    current = Path.cwd()
    for _ in range(5):
        if (current / "README.md").exists() or (current / ".git").exists():
            return current
        if current.name == "notebooks" and (current.parent.parent / "README.md").exists():
            return current.parent.parent
        current = current.parent
    return Path.cwd().parent.parent

BASE_PATH = find_project_root()
print(f"📁 Base path: {BASE_PATH.absolute()}")

📁 Base path: /Users/matias/Projetos/fiap/tech-challenge-1


In [2]:
# Carrega o arquivo parquet
parquet_file = BASE_PATH / "data/raw/VIOLBR24.parquet/4f70da4be33f4e138db43ae199f0c9af-0.parquet"

print(f"📂 Carregando arquivo: {parquet_file}")
df = pd.read_parquet(parquet_file)

print(f"\n✅ Dataset carregado com sucesso!")
print(f"📊 Dimensões: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"💾 Memória: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

📂 Carregando arquivo: /Users/matias/Projetos/fiap/tech-challenge-1/data/raw/VIOLBR24.parquet/4f70da4be33f4e138db43ae199f0c9af-0.parquet

✅ Dataset carregado com sucesso!
📊 Dimensões: 30,000 linhas × 160 colunas
💾 Memória: 240.75 MB


## Mapeamento de Colunas do Dicionário

O dicionário de dados do SINAN mapeia os nomes dos campos (DBF) para descrições. Vamos criar um dicionário de mapeamento baseado no documento oficial.

In [ ]:
# Dicionário de mapeamento baseado no documento oficial SINAN
# Formato: {nome_coluna_no_parquet: {descricao, categoria, tipo, obrigatorio}}
# Fonte: Dicionário de Dados SINAN NET - Versão 5.0/Patch 5.1

DICIONARIO_CAMPOS = {
    'NU_NOTIFIC': {
        'nome': 'N° da Notificação',
        'descricao': 'Número da Notificação - Campo Chave para identificação do registro no sistema',
        'tipo': 'varchar2(7)',
        'obrigatorio': True,
        'categoria': None
    },
    'TP_NOT': {
        'nome': 'Tipo de Notificação',
        'descricao': 'Identifica o tipo da notificação',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Negativa',
            '2': 'Individual',
            '3': 'Surto',
            '4': 'Agregado'
        }
    },
    'ID_AGRAVO': {
        'nome': 'Agravo',
        'descricao': 'Nome e código do agravo notificado segundo CID-10',
        'tipo': 'varchar2(4)',
        'obrigatorio': True,
        'categoria': 'Tabela de agravos do sistema com códigos CID-10'
    },
    'DT_NOTIFIC': {
        'nome': 'Data da Notificação',
        'descricao': 'Data de preenchimento da ficha de notificação (dd/mm/aaaa)',
        'tipo': 'date',
        'obrigatorio': True,
        'categoria': 'dd/mm/aaaa'
    },
    'SEM_NOT': {
        'nome': 'Semana epidemiológica da notificação',
        'descricao': 'Semanas do calendário epidemiológico padronizado (AAAASS)',
        'tipo': 'varchar2(6)',
        'obrigatorio': False,
        'categoria': 'Preenchida automaticamente'
    },
    'NU_ANO': {
        'nome': 'Ano da notificação',
        'descricao': 'Ano da notificação - Variável interna preenchida pelo sistema',
        'tipo': 'varchar(4)',
        'obrigatorio': False,
        'categoria': None
    },
    'SG_UF_NOT': {
        'nome': 'UF de Notificação',
        'descricao': 'Sigla da Unidade Federativa onde está localizada a unidade de saúde',
        'tipo': 'varchar2(2)',
        'obrigatorio': True,
        'categoria': 'Tabela com Códigos e siglas padronizados pelo IBGE'
    },
    'ID_MUNICIP': {
        'nome': 'Município de Notificação',
        'descricao': 'Código do município onde está localizada a unidade de saúde',
        'tipo': 'varchar2(6)',
        'obrigatorio': True,
        'categoria': 'Tabela com Código e nome dos municípios do cadastro do IBGE'
    },
    'ID_REGIONA': {
        'nome': 'Regional de Saúde',
        'descricao': 'Regional de saúde onde está localizado o município da unidade de saúde',
        'tipo': 'varchar2(4)',
        'obrigatorio': False,
        'categoria': 'Campo interno - Sistema relaciona o campo município de notificação'
    },
    'TP_UNI_EXT': {
        'nome': 'Unidade Notificadora',
        'descricao': 'Setor de atuação da unidade notificadora',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Unidade de Saúde',
            '2': 'Unidade de Assistência Social',
            '3': 'Estabelecimento de Ensino',
            '4': 'Conselho Tutelar',
            '5': 'Unidade de Saúde Indígena',
            '6': 'Centro Especializado de Atendimento à Mulher',
            '7': 'Outros'
        }
    },
    'CNES_NOT': {
        'nome': 'Unidade de Saúde - Código CNES',
        'descricao': 'Código do cadastro Nacional de Estabelecimento de Saúde (CNES)',
        'tipo': 'number(7,0)',
        'obrigatorio': True,
        'categoria': 'Código e nome da tabela do cadastro CNES'
    },
    'DT_OCOR': {
        'nome': 'Data da ocorrência da violência',
        'descricao': 'Data da ocorrência da violência (dd/mm/aaaa)',
        'tipo': 'date',
        'obrigatorio': True,
        'categoria': 'Data menor ou igual (<=) a Data de Notificação'
    },
    'SEM_PRI': {
        'nome': 'Semana epidemiológica dos primeiros sintomas',
        'descricao': 'Semanas do calendário epidemiológico padronizado (AAAASS)',
        'tipo': 'varchar2(6)',
        'obrigatorio': False,
        'categoria': 'Preenchida automaticamente'
    },
    'AUTOR_SEXO': {
        'nome': 'Sexo do provável autor da violência',
        'descricao': 'Informar o sexo do provável autor da agressão',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Masculino',
            '2': 'Feminino',
            '3': 'Ambos os sexos',
            '9': 'Ignorado'
        }
    },
    'AUTOR_ALCO': {
        'nome': 'Suspeita de uso de álcool',
        'descricao': 'Informar se o provável autor da agressão tinha suspeita de uso de alcool',
        'tipo': 'varchar2(1)',
        'obrigatorio': False,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'CICL_VID_AUTOR': {
        'nome': 'Ciclo de vida do Principal provável autor da violência',
        'descricao': 'Informar o ciclo de vida do provável autor da agressão',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Criança',
            '2': 'Adolescente',
            '3': 'Jovem',
            '4': 'Pessoa adulta',
            '5': 'Pessoa idosa',
            '9': 'Ignorado'
        }
    },
    'ENC_SAUDE': {
        'nome': 'Encaminhamento - Rede da Saúde',
        'descricao': 'Informar se houve encaminhamento no setor da rede da saúde',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'ASSIST_SOC': {
        'nome': 'Encaminhamento - Rede da Assistência Social',
        'descricao': 'Informar se houve encaminhamento no setor da rede Assistência Social',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'REDE_EDUCA': {
        'nome': 'Encaminhamento - Rede de Educação',
        'descricao': 'Informar se houve encaminhamento no setor da rede Educação',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'ATEND_MULH': {
        'nome': 'Encaminhamento - Rede de Atendimento à Mulher',
        'descricao': 'Informar se houve encaminhamento no setor da rede atendimento à mulher',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'CONS_TUTEL': {
        'nome': 'Encaminhamento - Conselho Tutelar',
        'descricao': 'Informar se houve encaminhamento para o Conselho Tutelar',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'CONS_IDO': {
        'nome': 'Encaminhamento - Conselho do Idoso',
        'descricao': 'Informar se houve encaminhamento para o Conselho do Idoso',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'DELEG_IDOSO': {
        'nome': 'Encaminhamento - Delegacia de Atendimento ao Idoso',
        'descricao': 'Informar se houve encaminhamento para delegacia de atendimento ao idoso',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'DIR_HUMAN': {
        'nome': 'Encaminhamento - Centro de Referência dos Direitos Humanos',
        'descricao': 'Informar se houve encaminhamento para centro de referência dos direitos Humanos',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'MPU': {
        'nome': 'Encaminhamento - Ministério Público',
        'descricao': 'Informar se houve encaminhamento para ministério Público',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'DELEG_CRIA': {
        'nome': 'Encaminhamento - Delegacia Especializada de Proteção à Criança e Adolescente',
        'descricao': 'Informar se houve encaminhamento para Delegacia Especializada de Proteção à Criança e Adolescente',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'DELEG_MULH': {
        'nome': 'Encaminhamento - Delegacia de Atendimento à Mulher',
        'descricao': 'Informar se houve encaminhamento para Delegacia de atendimento à mulher',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'DELEG': {
        'nome': 'Encaminhamento - Outras delegacias',
        'descricao': 'Informar se houve encaminhamento para outras delegacias',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'INFAN_JUV': {
        'nome': 'Encaminhamento - Justiça da infância e da Juventude',
        'descricao': 'Informar se houve encaminhamento para justiça da infância e da juventude',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'DEFEN_PUBL': {
        'nome': 'Encaminhamento - Defensoria pública',
        'descricao': 'Informar se houve encaminhamento para defensoria pública',
        'tipo': 'varchar2(1)',
        'obrigatorio': True,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'REL_TRAB': {
        'nome': 'Violência relacionada ao trabalho',
        'descricao': 'Informar se ocorreu violência relacionada ao trabalho',
        'tipo': 'varchar2(1)',
        'obrigatorio': False,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '9': 'Ignorado'
        }
    },
    'REL_CAT': {
        'nome': 'Comunicação de acidente de trabalho (CAT)',
        'descricao': 'Informar se foi emitida a CAT, caso a violência seja relacionada ao trabalho',
        'tipo': 'varchar2(1)',
        'obrigatorio': False,
        'categoria': {
            '1': 'Sim',
            '2': 'Não',
            '8': 'Não se aplica',
            '9': 'Ignorado'
        }
    },
    'CIRC_LESAO': {
        'nome': 'Circunstância da lesão',
        'descricao': 'Nome e código do agravo notificado segundo CID-10 - CAPITULO XX (VO1 a Y98)',
        'tipo': 'varchar2(5)',
        'obrigatorio': False,
        'categoria': 'Tabela de agravos do sistema com códigos CID-10'
    },
    'DT_ENCERRA': {
        'nome': 'Data de encerramento',
        'descricao': 'Data de encerramento do caso (dd/mm/aaaa)',
        'tipo': 'date',
        'obrigatorio': False,
        'categoria': 'Data >= data da notificação'
    },
    'DS_OBS': {
        'nome': 'Observações adicionais',
        'descricao': 'Observações adicionais sobre o caso',
        'tipo': 'text',
        'obrigatorio': False,
        'categoria': None
    }
}

print(f"📚 Dicionário carregado com {len(DICIONARIO_CAMPOS)} campos mapeados")
print(f"📖 Fonte: Dicionário de Dados SINAN NET - Versão 5.0/Patch 5.1")

📚 Dicionário carregado com 10 campos mapeados


## Análise de Cada Coluna

Abaixo apresentamos cada coluna do dataset com sua descrição, tipo, categorias possíveis e exemplos de valores encontrados.

In [4]:
def analisar_coluna(df, nome_coluna, info_dict=None):
    """
    Analisa uma coluna do dataframe e apresenta informações detalhadas.
    
    Parameters:
    -----------
    df : DataFrame
        DataFrame do pandas
    nome_coluna : str
        Nome da coluna a ser analisada
    info_dict : dict, optional
        Dicionário com informações do campo do SINAN
    """
    if nome_coluna not in df.columns:
        print(f"⚠️ Coluna '{nome_coluna}' não encontrada no dataset")
        return
    
    print("="*80)
    print(f"📋 COLUNA: {nome_coluna}")
    print("="*80)
    
    # Informações do dicionário
    if info_dict:
        print(f"\n📝 Nome Completo: {info_dict.get('nome', 'N/A')}")
        print(f"📖 Descrição: {info_dict.get('descricao', 'N/A')}")
        print(f"🔧 Tipo (SINAN): {info_dict.get('tipo', 'N/A')}")
        if info_dict.get('obrigatorio'):
            print("⚠️ Campo Obrigatório: Sim")
        else:
            print("⚠️ Campo Obrigatório: Não")
    
    # Informações do DataFrame
    print(f"\n🔍 Tipo (Pandas): {df[nome_coluna].dtype}")
    print(f"📊 Total de valores: {len(df[nome_coluna]):,}")
    
    # Valores nulos
    nulos = df[nome_coluna].isnull().sum()
    pct_nulos = (nulos / len(df[nome_coluna]) * 100)
    print(f"❌ Valores nulos: {nulos:,} ({pct_nulos:.2f}%)")
    print(f"✅ Valores preenchidos: {len(df[nome_coluna]) - nulos:,} ({100-pct_nulos:.2f}%)")
    
    # Valores únicos
    valores_unicos = df[nome_coluna].nunique()
    print(f"🔢 Valores únicos: {valores_unicos:,}")
    
    # Categorias (se houver no dicionário)
    if info_dict and 'categoria' in info_dict and isinstance(info_dict['categoria'], dict):
        print(f"\n📑 Categorias possíveis (segundo dicionário):")
        for codigo, descricao in info_dict['categoria'].items():
            count = (df[nome_coluna] == codigo).sum()
            pct = (count / len(df[nome_coluna]) * 100) if len(df[nome_coluna]) > 0 else 0
            print(f"   {codigo}: {descricao} ({count:,} ocorrências - {pct:.2f}%)")
    
    # Exemplos de valores
    print(f"\n💡 Exemplos de valores:")
    valores_nao_nulos = df[nome_coluna].dropna()
    if len(valores_nao_nulos) > 0:
        # Mostra até 5 valores únicos como exemplo
        exemplos = valores_nao_nulos.unique()[:5]
        for i, exemplo in enumerate(exemplos, 1):
            count = (df[nome_coluna] == exemplo).sum()
            print(f"   {i}. '{exemplo}' (aparece {count:,} vezes)")
    
    # Estatísticas (se numérico)
    if pd.api.types.is_numeric_dtype(df[nome_coluna]):
        print(f"\n📈 Estatísticas descritivas:")
        print(f"   Média: {df[nome_coluna].mean():.2f}")
        print(f"   Mediana: {df[nome_coluna].median():.2f}")
        print(f"   Mínimo: {df[nome_coluna].min()}")
        print(f"   Máximo: {df[nome_coluna].max()}")
        print(f"   Desvio padrão: {df[nome_coluna].std():.2f}")
    
    print("\n")

print("✅ Função de análise criada!")

✅ Função de análise criada!


In [5]:
# Lista todas as colunas do dataset
print("📋 TODAS AS COLUNAS DO DATASET:")
print("="*80)
colunas = sorted(df.columns.tolist())
for i, col in enumerate(colunas, 1):
    print(f"{i:3d}. {col}")
print(f"\nTotal: {len(colunas)} colunas")

📋 TODAS AS COLUNAS DO DATASET:
  1. AG_AMEACA
  2. AG_CORTE
  3. AG_ENFOR
  4. AG_ENVEN
  5. AG_ESPEC
  6. AG_FOGO
  7. AG_FORCA
  8. AG_OBJETO
  9. AG_OUTROS
 10. AG_QUENTE
 11. ANO_NASC
 12. ASSIST_SOC
 13. ATEND_MULH
 14. AUTOR_ALCO
 15. AUTOR_SEXO
 16. CICL_VID
 17. CIRC_LESAO
 18. CLASSI_FIN
 19. CONS_ABORT
 20. CONS_COMP
 21. CONS_DST
 22. CONS_ESPEC
 23. CONS_ESTRE
 24. CONS_GRAV
 25. CONS_IDO
 26. CONS_MENT
 27. CONS_OUTR
 28. CONS_SUIC
 29. CONS_TUTEL
 30. CS_ESCOL_N
 31. CS_GESTANT
 32. CS_RACA
 33. CS_SEXO
 34. DEFEN_PUBL
 35. DEF_AUDITI
 36. DEF_ESPEC
 37. DEF_FISICA
 38. DEF_MENTAL
 39. DEF_OUT
 40. DEF_TRANS
 41. DEF_VISUAL
 42. DELEG
 43. DELEG_CRIA
 44. DELEG_IDOS
 45. DELEG_MULH
 46. DIR_HUMAN
 47. DT_DIGITA
 48. DT_ENCERRA
 49. DT_INVEST
 50. DT_NOTIFIC
 51. DT_OBITO
 52. DT_OCOR
 53. DT_TRANSDM
 54. DT_TRANSRM
 55. DT_TRANSRS
 56. DT_TRANSSE
 57. DT_TRANSSM
 58. DT_TRANSUS
 59. ENC_ABRIGO
 60. ENC_CREAS
 61. ENC_DEAM
 62. ENC_DELEG
 63. ENC_DPCA
 64. ENC_ESPEC
 65. E

In [6]:
# Analisa colunas que temos no dicionário
colunas_mapeadas = [col for col in DICIONARIO_CAMPOS.keys() if col in df.columns]

print(f"🔍 Analisando {len(colunas_mapeadas)} colunas mapeadas no dicionário...\n")

for coluna in colunas_mapeadas:
    analisar_coluna(df, coluna, DICIONARIO_CAMPOS[coluna])

🔍 Analisando 8 colunas mapeadas no dicionário...

📋 COLUNA: TP_NOT

📝 Nome Completo: Tipo de Notificação
📖 Descrição: Identifica o tipo da notificação
🔧 Tipo (SINAN): varchar2(1)
⚠️ Campo Obrigatório: Sim

🔍 Tipo (Pandas): object
📊 Total de valores: 30,000
❌ Valores nulos: 0 (0.00%)
✅ Valores preenchidos: 30,000 (100.00%)
🔢 Valores únicos: 1

📑 Categorias possíveis (segundo dicionário):
   1: Negativa (0 ocorrências - 0.00%)
   2: Individual (30,000 ocorrências - 100.00%)
   3: Surto (0 ocorrências - 0.00%)
   4: Agregado (0 ocorrências - 0.00%)

💡 Exemplos de valores:
   1. '2' (aparece 30,000 vezes)


📋 COLUNA: ID_AGRAVO

📝 Nome Completo: Agravo
📖 Descrição: Nome e código do agravo notificado segundo CID-10
🔧 Tipo (SINAN): varchar2(4)
⚠️ Campo Obrigatório: Sim

🔍 Tipo (Pandas): object
📊 Total de valores: 30,000
❌ Valores nulos: 0 (0.00%)
✅ Valores preenchidos: 30,000 (100.00%)
🔢 Valores únicos: 1

💡 Exemplos de valores:
   1. 'Y09 ' (aparece 30,000 vezes)


📋 COLUNA: DT_NOTIFIC

📝 No

### Análise de Todas as Colunas (Resumo)

Abaixo apresentamos um resumo de todas as colunas do dataset, mesmo aquelas que não estão mapeadas no dicionário.

In [7]:
# Cria um resumo de todas as colunas
resumo_colunas = []

for col in df.columns:
    info = {
        'Coluna': col,
        'Tipo': str(df[col].dtype),
        'Valores Únicos': df[col].nunique(),
        'Valores Nulos': df[col].isnull().sum(),
        '% Nulos': f"{(df[col].isnull().sum() / len(df) * 100):.2f}%",
        'Tem no Dicionário': 'Sim' if col in DICIONARIO_CAMPOS else 'Não'
    }
    
    # Adiciona exemplo de valor
    valores_nao_nulos = df[col].dropna()
    if len(valores_nao_nulos) > 0:
        exemplo = valores_nao_nulos.iloc[0]
        if isinstance(exemplo, str) and len(str(exemplo)) > 30:
            info['Exemplo'] = str(exemplo)[:30] + "..."
        else:
            info['Exemplo'] = str(exemplo)
    else:
        info['Exemplo'] = "N/A"
    
    resumo_colunas.append(info)

df_resumo = pd.DataFrame(resumo_colunas)
df_resumo

,Coluna,Tipo,Valores Únicos,Valores Nulos,% Nulos,Tem no Dicionário,Exemplo
0,TP_NOT,object,1,0,0.00%,Sim,2
1,ID_AGRAVO,object,1,0,0.00%,Sim,Y09
2,DT_NOTIFIC,object,192,0,0.00%,Sim,20240627
3,SEM_NOT,object,28,0,0.00%,Sim,202426
4,NU_ANO,object,1,0,0.00%,Sim,2024
...,...,...,...,...,...,...,...
155,DELEG_MULH,object,4,0,0.00%,Não,2
156,DELEG,object,4,0,0.00%,Não,2
157,INFAN_JUV,object,4,0,0.00%,Não,2
158,DEFEN_PUBL,object,4,0,0.00%,Não,2


### Análise Detalhada de Colunas Selecionadas

Vamos analisar algumas colunas específicas com mais detalhes, incluindo exemplos reais dos dados.

In [8]:
# Função para análise detalhada com exemplos
def analisar_coluna_detalhada(df, nome_coluna, n_exemplos=10):
    """Analisa uma coluna mostrando exemplos reais dos dados."""
    if nome_coluna not in df.columns:
        print(f"⚠️ Coluna '{nome_coluna}' não encontrada")
        return
    
    print("="*80)
    print(f"📋 ANÁLISE DETALHADA: {nome_coluna}")
    print("="*80)
    
    # Informações básicas
    print(f"\n📊 Informações Gerais:")
    print(f"   Tipo: {df[nome_coluna].dtype}")
    print(f"   Total de registros: {len(df):,}")
    print(f"   Valores únicos: {df[nome_coluna].nunique():,}")
    print(f"   Valores nulos: {df[nome_coluna].isnull().sum():,} ({(df[nome_coluna].isnull().sum()/len(df)*100):.2f}%)")
    
    # Distribuição de valores (top 10)
    print(f"\n📈 Top 10 valores mais frequentes:")
    top_valores = df[nome_coluna].value_counts().head(10)
    for valor, count in top_valores.items():
        pct = (count / len(df) * 100)
        print(f"   '{valor}': {count:,} vezes ({pct:.2f}%)")
    
    # Exemplos aleatórios
    print(f"\n💡 Exemplos aleatórios de valores (primeiros {n_exemplos}):")
    valores_nao_nulos = df[nome_coluna].dropna()
    if len(valores_nao_nulos) > 0:
        exemplos = valores_nao_nulos.head(n_exemplos)
        for i, exemplo in enumerate(exemplos, 1):
            print(f"   {i:2d}. {exemplo}")
    
    print("\n")

# Analisa algumas colunas importantes
colunas_importantes = ['TP_NOT', 'SG_UF_NOT', 'ID_MUNICIP', 'DT_NOTIFIC', 'DT_OCOR']

for col in colunas_importantes:
    if col in df.columns:
        analisar_coluna_detalhada(df, col)

📋 ANÁLISE DETALHADA: TP_NOT

📊 Informações Gerais:
   Tipo: object
   Total de registros: 30,000
   Valores únicos: 1
   Valores nulos: 0 (0.00%)

📈 Top 10 valores mais frequentes:
   '2': 30,000 vezes (100.00%)

💡 Exemplos aleatórios de valores (primeiros 10):
    1. 2
    2. 2
    3. 2
    4. 2
    5. 2
    6. 2
    7. 2
    8. 2
    9. 2
   10. 2


📋 ANÁLISE DETALHADA: SG_UF_NOT

📊 Informações Gerais:
   Tipo: object
   Total de registros: 30,000
   Valores únicos: 27
   Valores nulos: 0 (0.00%)

📈 Top 10 valores mais frequentes:
   '32': 9,817 vezes (32.72%)
   '35': 4,503 vezes (15.01%)
   '33': 2,423 vezes (8.08%)
   '31': 1,984 vezes (6.61%)
   '41': 1,846 vezes (6.15%)
   '23': 1,192 vezes (3.97%)
   '43': 1,047 vezes (3.49%)
   '26': 1,007 vezes (3.36%)
   '29': 827 vezes (2.76%)
   '42': 773 vezes (2.58%)

💡 Exemplos aleatórios de valores (primeiros 10):
    1. 41
    2. 41
    3. 41
    4. 41
    5. 33
    6. 35
    7. 41
    8. 33
    9. 33
   10. 33


📋 ANÁLISE DETALHADA: 

### Visualização de Exemplos de Registros

Vamos visualizar alguns registros completos para entender melhor a estrutura dos dados.

In [9]:
# Mostra primeiras linhas
print("📋 PRIMEIRAS 5 LINHAS DO DATASET:")
print("="*80)
df.head()

📋 PRIMEIRAS 5 LINHAS DO DATASET:


,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_UNIDADE,DT_OCOR,SEM_PRI,...,CONS_IDO,DELEG_IDOS,DIR_HUMAN,MPU,DELEG_CRIA,DELEG_MULH,DELEG,INFAN_JUV,DEFEN_PUBL,DT_ENCERRA
0,2,Y09,20240627,202426,2024,41,411370,2579332,20240627,202426,...,2,2,2,2,2,2,2,2,2,20240711
1,2,Y09,20240627,202426,2024,41,411370,2579332,20240627,202426,...,2,2,2,2,2,2,2,2,2,20240711
2,2,Y09,20240627,202426,2024,41,411370,2579332,20240627,202426,...,2,2,2,2,2,2,2,2,2,20240708
3,2,Y09,20240627,202426,2024,41,411370,2579332,20240627,202426,...,2,2,2,2,2,2,2,2,2,20240711
4,2,Y09,20240627,202426,2024,33,330100,2287579,20240627,202426,...,2,2,2,2,2,2,2,2,2,20240627


In [10]:
# Mostra informações gerais
print("📊 INFORMAÇÕES GERAIS DO DATASET:")
print("="*80)
df.info()

📊 INFORMAÇÕES GERAIS DO DATASET:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Columns: 160 entries, TP_NOT to DT_ENCERRA
dtypes: object(160)
memory usage: 36.6+ MB


### Resumo Final

Este notebook apresentou uma análise das colunas do dataset SINAN - Violência Interpessoal/Autoprovocada. 

**Próximos passos sugeridos:**
- Limpeza de dados (valores nulos, inconsistências)
- Análise exploratória de dados (EDA)
- Feature engineering
- Modelagem preditiva

**Referências:**
- Dicionário de Dados SINAN NET - Versão 5.0/Patch 5.1
- Documento disponível em: `docs/dicionario_dados_sinan.pdf`